# 03 · Tratamento e Limpeza dos Dados — Hit Maker (Spotify)

Módulo anterior: [02 · Requisitos](02_requisitos.ipynb). Próximo módulo: [04 · Ideação da Solução](04_ideacao_solucao.ipynb).

Implementa a **Frente 1** das GQs refinadas (Auditoria e Diagnóstico do Dataset). Código reaproveitado do pipeline já escrito (`hitmakerprjct` / Colab), rodando sobre `archive/dataset.csv` — a única fonte com o schema completo (ver [02 · Requisitos](02_requisitos.ipynb)).

Ao final, salva o dataframe tratado em `artifacts/df_limpo.csv` para os módulos seguintes.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
RANDOM_STATE = 42

## Configuração

`COLUNAS` mapeia os nomes "de negócio" para os nomes reais das colunas no CSV — trocar de fonte de dados exige ajustar só este dicionário (requisito de negócio nº 2 do módulo 02).

In [ ]:
# ATENÇÃO: confira estes nomes contra o CSV real. Datasets do Spotify costumam
# variar (ex: 'track_genre' vs 'genre', 'popularity' vs 'track_popularity').
# Ajuste o dicionário abaixo em UM lugar só, e o resto do notebook se adapta.
COLUNAS = {
    "genero": "track_genre",
    "popularidade": "popularity",
    "id_faixa": "track_id",
    "nome_faixa": "track_name",
    "artista": "artists",
}

FEATURES_NUMERICAS = [
    "danceability", "energy", "valence", "tempo", "loudness",
    "acousticness", "instrumentalness", "liveness", "speechiness",
]

DATASET_PATH = Path("../archive/dataset.csv")
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

## Frente 1: Auditoria e Diagnóstico do Dataset

GQ: *O dataset possui valores ausentes, duplicatas ou outliers críticos nas variáveis principais que comprometam a confiabilidade da análise? A distribuição de faixas por gênero é equilibrada ou assimétrica?*

In [ ]:
def carregar_dataset(caminho_csv) -> pd.DataFrame:
    '''Carrega o dataset bruto do Kaggle.'''
    df = pd.read_csv(caminho_csv)
    # ATENÇÃO: este CSV específico (archive/dataset.csv) tem uma coluna de
    # índice remanescente do pandas (primeira coluna sem nome) — removida aqui
    # para não ser confundida com uma feature real.
    primeira_coluna = df.columns[0]
    if primeira_coluna.startswith("Unnamed"):
        df = df.drop(columns=[primeira_coluna])
    return df


df_bruto = carregar_dataset(DATASET_PATH)
df_bruto.shape

In [ ]:
def auditoria_dataset(df: pd.DataFrame) -> dict:
    """
    GQ: 'O dataset possui valores ausentes, duplicatas ou outliers críticos
    nas variáveis principais que comprometam a confiabilidade da análise?'
    """
    relatorio = {}
    relatorio["shape"] = df.shape
    relatorio["n_generos"] = df[COLUNAS["genero"]].nunique()
    relatorio["valores_ausentes"] = df.isnull().sum().sort_values(ascending=False)
    relatorio["duplicatas"] = df.duplicated().sum()
    # ATENÇÃO: duplicidade "real" costuma ser por (artista + nome da faixa),
    # não pela linha inteira — vale checar as duas formas.
    relatorio["duplicatas_artista_faixa"] = df.duplicated(
        subset=[COLUNAS["artista"], COLUNAS["nome_faixa"]]
    ).sum()
    relatorio["dtypes"] = df.dtypes
    relatorio["describe"] = df[FEATURES_NUMERICAS + [COLUNAS["popularidade"]]].describe()
    return relatorio


relatorio_auditoria = auditoria_dataset(df_bruto)
print("Shape:", relatorio_auditoria["shape"])
print("Nº de gêneros:", relatorio_auditoria["n_generos"])
print("Duplicatas (linha inteira):", relatorio_auditoria["duplicatas"])
print("Duplicatas (artista+faixa):", relatorio_auditoria["duplicatas_artista_faixa"])
relatorio_auditoria["valores_ausentes"]

In [ ]:
relatorio_auditoria["describe"]

In [ ]:
def distribuicao_generos(df: pd.DataFrame):
    """GQ: distribuição de faixas por gênero — equilibrada ou assimétrica?"""
    freq = df[COLUNAS["genero"]].value_counts(normalize=True).sort_values(ascending=False)
    plt.figure(figsize=(10, 8))
    freq.head(30).plot(kind="barh")
    plt.title("Distribuição relativa de faixas por gênero (top 30)")
    plt.xlabel("Proporção do dataset")
    plt.tight_layout()
    return freq


freq_generos = distribuicao_generos(df_bruto)

In [ ]:
def detectar_outliers_iqr(df: pd.DataFrame, colunas: list) -> pd.DataFrame:
    """Detecta outliers via IQR (1.5x) para as colunas numéricas informadas."""
    contagem = {}
    for col in colunas:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        limite_inf, limite_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        contagem[col] = ((df[col] < limite_inf) | (df[col] > limite_sup)).sum()
    return pd.Series(contagem).sort_values(ascending=False)


detectar_outliers_iqr(df_bruto, FEATURES_NUMERICAS + [COLUNAS["popularidade"]])

## Tratamento

Decisão de negócio nº 3 do módulo 02: aqui optamos por **imputação pela mediana** (estratégia `"mediana"`) como padrão — preserva o volume de faixas, e a mediana é robusta a outliers nas features de áudio. A alternativa `"remover"` está implementada e disponível caso o time decida por outro caminho.

In [ ]:
def tratar_valores_ausentes(df: pd.DataFrame, estrategia: str = "mediana") -> pd.DataFrame:
    """
    Aplica a estratégia de tratamento de nulos.
    # ATENÇÃO: decisão de negócio — imputar, remover ou preservar?
    Documente a escolha e a justificativa junto com o time.
    """
    df = df.copy()
    if estrategia == "mediana":
        for col in FEATURES_NUMERICAS:
            if col in df.columns:
                df[col] = df[col].fillna(df[col].median())
    elif estrategia == "remover":
        df = df.dropna(subset=FEATURES_NUMERICAS)
    return df


df_limpo = tratar_valores_ausentes(df_bruto, estrategia="mediana")
print("Nulos restantes:", df_limpo[FEATURES_NUMERICAS].isnull().sum().sum())
df_limpo.shape

## Artefato de saída

Salva o dataframe tratado para os módulos 04 e 05 lerem, sem precisar reexecutar a limpeza.

In [ ]:
df_limpo.to_csv(ARTIFACTS_DIR / "df_limpo.csv", index=False)
print("Salvo em:", (ARTIFACTS_DIR / "df_limpo.csv").resolve())

## Resumo do módulo

- Dataset carregado: `archive/dataset.csv` (~114 mil faixas, coluna de índice residual removida).
- Auditoria feita: nulos por coluna, duplicatas (linha inteira e artista+faixa), outliers via IQR, describe das features numéricas + popularidade.
- Nulos nas features numéricas tratados por mediana (decisão documentada acima).
- **Pendências que ficam para o time validar** (não resolvidas neste módulo): o que fazer com as duplicatas encontradas, e se os outliers detectados devem ser removidos/capados ou mantidos — ver decisões 3–4 do módulo [02 · Requisitos](02_requisitos.ipynb).

Próximo módulo: [04 · Ideação da Solução](04_ideacao_solucao.ipynb).